# LightGCN Implementation for Recommendation

https://medium.com/@jn2279/better-recommender-systems-with-lightgcn-a0e764af14f9

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch_geometric.nn.conv import MessagePassing
from torch_geometric.utils import degree
from tqdm.notebook import tqdm
import random

from helpers.data_loaders import load_movielens_data, load_steam_data

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")


Using device: cuda


## 2. Data Loading and Preparation

In [ ]:
movies_df, ratings_df = load_movielens_data()
reviews_df_steam, items_df_steam = load_steam_data()

ratings_df = ratings_df[ratings_df['rating'] >= 4.0]
movielens_interactions = pd.DataFrame({
    'user_id': 'movielens_user_' + ratings_df['userId'].astype(str),
    'item_id': 'movielens_item_' + ratings_df['movieId'].astype(str)
})

steam_interactions = pd.DataFrame({
    'user_id': 'steam_user_' + reviews_df_steam['user_id'].astype(str),
    'item_id': 'steam_item_' + reviews_df_steam['app_id'].astype(str)
})

all_interactions = pd.concat([movielens_interactions, steam_interactions]).drop_duplicates()

print(f"Total unique interactions: {len(all_interactions)}")

unique_users = all_interactions['user_id'].unique()
unique_items = all_interactions['item_id'].unique()

user_map = {user: i for i, user in enumerate(unique_users)}
item_map = {item: i for i, item in enumerate(unique_items)}

num_users = len(user_map)
num_items = len(item_map)

print(f"Number of users: {num_users}")
print(f"Number of items: {num_items}")

all_interactions['user_idx'] = all_interactions['user_id'].map(user_map)
all_interactions['item_idx'] = all_interactions['item_id'].map(item_map)

user_indices = torch.LongTensor(all_interactions['user_idx'].values)
item_indices = torch.LongTensor(all_interactions['item_idx'].values)

edge_index = torch.stack([
    torch.cat([user_indices, item_indices + num_users]),
    torch.cat([item_indices + num_users, user_indices])
], dim=0)

print("Edge index created:")
print(edge_index.shape)


Total unique interactions: 12497401
Number of users: 184419
Number of items: 43660
Edge index created:
torch.Size([2, 24994802])


## 3. LightGCN Model Definition

In [3]:
class LightGCN(MessagePassing):
    def __init__(self, num_users, num_items, embedding_dim=64, num_layers=3): 
        super().__init__(aggr='add')
        self.num_users = num_users
        self.num_items = num_items
        self.embedding_dim = embedding_dim
        self.num_layers = num_layers

        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)

        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)

    def forward(self, edge_index):
        # Initial embeddings (E^0)
        x = torch.cat([self.user_embedding.weight, self.item_embedding.weight])
        
        # Calculate normalization for symmetric adjacency matrix
        row, col = edge_index
        deg = degree(col, x.size(0), dtype=x.dtype)
        deg_inv_sqrt = deg.pow(-0.5)
        deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]
        
        # Propagation loop
        final_embs = (1 / (self.num_layers + 1)) * x
        
        for _ in range(self.num_layers):
            x = self.propagate(edge_index, x=x, norm=norm)
            final_embs += (1 / (self.num_layers + 1)) * x
            
        user_final_embs, item_final_embs = torch.split(final_embs, [self.num_users, self.num_items])
        
        return user_final_embs, item_final_embs

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j


## 4. Training Setup

In [ ]:
class BPRDataset(Dataset):
    def __init__(self, interactions, num_items, user_map): 
        self.interactions = interactions
        self.num_items = num_items
        self.user_map = user_map
        
        self.user_pos_items = self.interactions.groupby('user_idx')['item_idx'].apply(set)

    def __len__(self):
        return len(self.interactions)

    def __getitem__(self, idx):
        interaction = self.interactions.iloc[idx]
        user_idx = interaction['user_idx']
        pos_item_idx = interaction['item_idx']
        
        neg_item_idx = random.randint(0, self.num_items - 1)
        while neg_item_idx in self.user_pos_items[user_idx]:
            neg_item_idx = random.randint(0, self.num_items - 1)
            
        return user_idx, pos_item_idx, neg_item_idx

def bpr_loss(users_emb, pos_items_emb, neg_items_emb, users_emb_0, pos_items_emb_0, neg_items_emb_0, lambda_reg=1e-4): 
    pos_scores = torch.sum(users_emb * pos_items_emb, dim=1)
    neg_scores = torch.sum(users_emb * neg_items_emb, dim=1)
    
    loss = -torch.mean(torch.nn.functional.logsigmoid(pos_scores - neg_scores))
    
    reg_loss = (users_emb_0.norm(2).pow(2) + 
               pos_items_emb_0.norm(2).pow(2) + 
               neg_items_emb_0.norm(2).pow(2)) / 2
               
    return loss + lambda_reg * reg_loss

embedding_dim = 32
num_layers = 3
batch_size = 4096
learning_rate = 1e-3
epochs = 2

train_dataset = BPRDataset(all_interactions, num_items, user_map)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

model = LightGCN(num_users, num_items, embedding_dim, num_layers).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
edge_index = edge_index.to(device)


## 5. Training Loop

In [ ]:
model.train()
for epoch in range(epochs):
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    
    for user_batch, pos_item_batch, neg_item_batch in progress_bar:
        optimizer.zero_grad()
        
        user_final_embs, item_final_embs = model(edge_index)
        
        user_embs = user_final_embs[user_batch.to(device)]
        pos_item_embs = item_final_embs[pos_item_batch.to(device)]
        neg_item_embs = item_final_embs[neg_item_batch.to(device)]
        
        user_embs_0 = model.user_embedding(user_batch.to(device))
        pos_item_embs_0 = model.item_embedding(pos_item_batch.to(device))
        neg_item_embs_0 = model.item_embedding(neg_item_batch.to(device))
        
        loss = bpr_loss(user_embs, pos_item_embs, neg_item_embs, 
                        user_embs_0, pos_item_embs_0, neg_item_embs_0)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item()})
        
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs}, Average Loss: {avg_loss:.4f}")


Epoch 1/2:   0%|          | 0/3052 [00:00<?, ?it/s]

Epoch 1/2, Average Loss: 0.4137


Epoch 2/2:   0%|          | 0/3052 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 6. Making Recommendations

In [22]:
def get_recommendations(user_id_str, model, top_k=10):
    model.eval()
    
    if user_id_str not in user_map:
        print(f"User '{user_id_str}' not found.")
        return
    user_idx = user_map[user_id_str]
    
    with torch.no_grad():
        user_final_embs, item_final_embs = model(edge_index)
        
        user_emb = user_final_embs[user_idx]
        
        scores = torch.matmul(user_emb, item_final_embs.T)
        
        top_k_scores, top_k_indices = torch.topk(scores, k=top_k)
        
        inv_item_map = {i: item for item, i in item_map.items()}
        
        print(f"Top {top_k} recommendations for user '{user_id_str}':")
        for i, score in zip(top_k_indices.cpu().numpy(), top_k_scores.cpu().numpy()):
            if 'steam' in inv_item_map[i]:
                print(f"  - [steam]Item: {inv_item_map[i]}, Title: {items_df_steam.loc[items_df_steam['app_id'] == int(inv_item_map[i].split('_')[-1]), 'title'].values[0]} , Score: {score:.4f}")
            else:
                print(f"  - [movie]Item: {inv_item_map[i]}, Title: {movies_df.loc[movies_df['movieId'] == int(inv_item_map[i].split('_')[-1]), 'title'].values[0]} , Score: {score:.4f}")

        print(f"Least preferred {top_k} items for user '{user_id_str}':")
        bottom_k_scores, bottom_k_indices = torch.topk(scores, k=top_k, largest=False)
        for i, score in zip(bottom_k_indices.cpu().numpy(), bottom_k_scores.cpu().numpy()):
            if 'steam' in inv_item_map[i]:
                print(f"  - [steam]Item: {inv_item_map[i]}, Title: {items_df_steam.loc[items_df_steam['app_id'] == int(inv_item_map[i].split('_')[-1]), 'title'].values[0]} , Score: {score:.4f}")
            else:
                print(f"  - [movie]Item: {inv_item_map[i]}, Title: {movies_df.loc[movies_df['movieId'] == int(inv_item_map[i].split('_')[-1]), 'title'].values[0]} , Score: {score:.4f}")

sample_user_id = 'movielens_user_2'
print(f"Items liked by the user ({sample_user_id}):")

liked_items = all_interactions[all_interactions['user_id'] == sample_user_id]['item_id']
for item in liked_items:
    print(f"  - {item}, Title: {movies_df.loc[movies_df['movieId'] == int(item.split('_')[-1]), 'title'].values[0]}")

get_recommendations(sample_user_id, model)


Items liked by the user (movielens_user_2):
  - movielens_item_110, Title: Braveheart (1995)
  - movielens_item_150, Title: Apollo 13 (1995)
  - movielens_item_151, Title: Rob Roy (1995)
  - movielens_item_236, Title: French Kiss (1995)
  - movielens_item_260, Title: Star Wars: Episode IV - A New Hope (1977)
  - movielens_item_318, Title: Shawshank Redemption, The (1994)
  - movielens_item_333, Title: Tommy Boy (1995)
  - movielens_item_349, Title: Clear and Present Danger (1994)
  - movielens_item_356, Title: Forrest Gump (1994)
  - movielens_item_364, Title: Lion King, The (1994)
  - movielens_item_457, Title: Fugitive, The (1993)
  - movielens_item_497, Title: Much Ado About Nothing (1993)
  - movielens_item_527, Title: Schindler's List (1993)
  - movielens_item_534, Title: Shadowlands (1993)
  - movielens_item_589, Title: Terminator 2: Judgment Day (1991)
  - movielens_item_733, Title: Rock, The (1996)
  - movielens_item_914, Title: My Fair Lady (1964)
  - movielens_item_953, Title

idk how to connect?

Why promising?
- Featureless - only interaction matters 
- no feature transformation, no nonlinear activation
- only learns embeddings and propagates them